# Get Papers

In [ ]:
import os
import urllib.parse
import urllib.request
import xml.etree.ElementTree as ET

PAPERS = [
    "Are Bias Mitigation Techniques for Deep Learning Effective?",
    "Operationalizing the EU AI Act in Agile Software Development: A Guideline-Based Approach",
    "From Obligation to Specification: A Survey on Validating EU AI Act Requirements in RE",
    "Mapping the Regulatory Learning Space for the EU AI Act",
    "AI Governance in the Context of the EU AI Act: A Bibliometric and Literature Review Approach",
    "Red Teaming AI Policy: A Taxonomy of Avoision and the EU AI Act",
    "Navigating the EU AI Act: A Methodological Approach to Compliance for Safety-critical Products",
    "ADAPT Centre Contribution on Implementation of the EU AI Act and Fundamental Right Protection",
    "From Bias to Accountability: How the EU AI Act Confronts Challenges in European GeoAI Auditing",
    "Complying with the EU AI Act",
    "The EU AI Act in Development Practice: A Pro-justice Approach",
    "Qualifying and Quantifying Risk Under the EU AI Act",
    "The Case for ESM3 as a General-Purpose AI Model with Systemic Risk Under the EU AI Act",
    "An Analysis of the New EU AI Act and A Proposed Standardization Framework for Machine Learning Fairness",
    "The EU AI Act and the Rights-based Approach to Technological Governance",
    "Assessing Model-Agnostic XAI Methods against EU AI Act Explainability Requirements",
    "Complying with the EU AI Act: Innovations in Explainable and User-Centric Hand Gesture Recognition",
    "Sustainable AI Regulation",
    "Governing What the EU AI Act Excludes: Accountability for Autonomous AI Agents in Smart City Critical Infrastructure",
    "Equality of Opportunity in Supervised Learning" # The famous Equalized Odds paper!
]

os.makedirs("papers", exist_ok=True)

for i, title in enumerate(PAPERS, 1):
    print(f"[{i}/{len(PAPERS)}] Searching for: {title}...")
    query = urllib.parse.quote(f'ti:"{title}"')
    url = f'http://export.arxiv.org/api/query?search_query={query}&max_results=1'

    try:
        req = urllib.request.urlopen(url)
        xml_data = req.read()
        root = ET.fromstring(xml_data)

        # Parse arXiv response
        entry = root.find('{http://www.w3.org/2005/Atom}entry')
        if entry is not None:
            id_url = entry.find('{http://www.w3.org/2005/Atom}id').text
            arxiv_id = id_url.split('/abs/')[-1]
            pdf_url = f'https://arxiv.org/pdf/{arxiv_id}.pdf'

            filename = os.path.join("papers", f"{arxiv_id}.pdf")
            print(f"  --> Found on arXiv! Downloading {pdf_url}...")
            urllib.request.urlretrieve(pdf_url, filename)
            print("  --> Saved successfully.")
        else:
            print("  --> Not found via exact arXiv API query (may be on ACM/Springer or publisher page).")
    except Exception as e:
        print(f"  --> Error: {e}")

print("\nDone! Check your papers/ folder.")

# Main Code

In [ ]:
!pip install pymupdf langchain langchain-text-splitters \
             chromadb sentence-transformers groq tqdm python-dotenv \
             -q

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

# your papers folder
PAPERS_DIR = "/content/drive/MyDrive/governance_rag_project/governance_rag_papers"

In [ ]:
from google.colab import userdata
import fitz #pymupdf
import os, json, re, time
from pathlib import Path
from collections import Counter
import chromadb
from sentence_transformers import SentenceTransformer, CrossEncoder
from langchain_text_splitters import RecursiveCharacterTextSplitter
from tqdm import tqdm
from groq import Groq
import warnings
import os
import logging
logging.getLogger("sentence_transformers").setLevel(logging.ERROR)
warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"


CHUNKS_FILE = "/content/drive/MyDrive/governance_rag_project/chunks.json"

In [ ]:
# Configs
PAPERS_DIR      = "/content/drive/MyDrive/governance_rag_project/governance_rag_papers"
CHUNKS_FILE     = "/content/drive/MyDrive/governance_rag_project/chunks.json"
CHROMA_DIR      = "/content/drive/MyDrive/governance_rag_project/chroma_db"
EMBEDDING_MODEL = "all-MiniLM-L6-v2"
RERANK_MODEL    = "cross-encoder/ms-marco-MiniLM-L-6-v2"
GROQ_MODEL      = "llama-3.1-8b-instant"
COLLECTION_NAME = "governance_rag_papers"
CHUNK_SIZE      = 1000
CHUNK_OVERLAP   = 200
BATCH_SIZE      = 64
RETRIEVE_N      = 20
TOP_K           = 5
MAX_PER_PAPER   = 2

GROQ_API_KEY    = userdata.get('rag_api_key')


print("Config ready.")

In [ ]:
def clean_text(text):
    """Remove common PDF junk — headers, footers, excessive whitespace."""
    # join hyphenated line breaks e.g. "discrimina-\ntory" → "discriminatory"
    text = re.sub(r'(\w+)-\s*\n\s*(\w+)', r'\1\2', text)
    # collapse multiple newlines
    text = re.sub(r'\n{3,}', '\n\n', text)
    # collapse multiple spaces
    text = re.sub(r' {2,}', ' ', text)
    # remove lines that are just numbers (page numbers)
    text = re.sub(r'^\s*\d+\s*$', '', text, flags=re.MULTILINE)
    # remove leading period leftover from sentence splitting
    text = re.sub(r'^\.\s*', '', text)
    # remove references section — everything after "References" or "Bibliography"
    text = re.sub(r'\n(References|Bibliography|REFERENCES)\n.*', '', text, flags=re.DOTALL)
    # remove lines shorter than 60 chars that repeat more than 3 times
    # these are usually headers/footers
    lines = text.split('\n')
    line_counts = Counter(lines)
    text = '\n'.join(
        line for line in lines
        if line_counts[line] <= 3 or len(line) > 60
    )
    return text.strip()

pdf_files = list(Path(PAPERS_DIR).glob("*.pdf"))
print(f"Found {len(pdf_files)} PDFs\n")

documents = []  # each item = {text, filename, title}

for pdf_path in pdf_files:
    try:
        doc  = fitz.open(str(pdf_path))
        text = ""
        for page in doc:
            text += page.get_text()
        doc.close()

        text = clean_text(text)

        if len(text) < 500:
            print(f"⚠ Skipping {pdf_path.name} — too little text extracted")
            continue

        documents.append({
            "filename": pdf_path.name,
            "title":    pdf_path.stem,  # filename without .pdf
            "text":     text,
        })
        print(f"✓ {pdf_path.name} — {len(text):,} characters")

    except Exception as e:
        print(f"✗ Failed: {pdf_path.name} — {e}")

print(f"\nSuccessfully extracted: {len(documents)} documents")

In [ ]:
# chunk_size: how many characters per chunk
# chunk_overlap: how many characters overlap between chunks
#   (overlap is important — it prevents losing context at boundaries)

splitter = RecursiveCharacterTextSplitter(
    chunk_size    = 1000,
    chunk_overlap = 200,
    separators = ["\n\n", ". ", "\n", " ", ""]
)

all_chunks = []

for doc in documents:
    chunks = splitter.split_text(doc["text"])

    for i, chunk_text in enumerate(chunks):
    # clean up any leading period/whitespace left over from sentence splitting
      chunk_text = re.sub(r'^[\.\s]+', '', chunk_text).strip()
      all_chunks.append({
          "chunk_id":  f"{doc['filename']}__chunk_{i:04d}",
          "filename":  doc["filename"],
          "title":     doc["title"],
          "chunk_idx": i,
          "text":      chunk_text,
      })

print(f"Total chunks: {len(all_chunks)}")
print(f"Average chunk size: {sum(len(c['text']) for c in all_chunks) // len(all_chunks)} characters")
print(f"Chunks per document (average): {len(all_chunks) // len(documents)}")


In [ ]:
sample = all_chunks[21]  # pick chunk #10 as a sample
print("=== SAMPLE CHUNK ===")
print(f"From:  {sample['title']}")
print(f"Chunk: {sample['chunk_idx']}")
print(f"Size:  {len(sample['text'])} characters")
print(f"\n{sample['text']}")

In [ ]:
with open(CHUNKS_FILE, "w") as f:
    json.dump(all_chunks, f, indent=2)

print(f"\nSaved {len(all_chunks)} chunks to {CHUNKS_FILE}")

In [ ]:
with open(CHUNKS_FILE, "r") as f:
    all_chunks = json.load(f)

print(f"Loaded {len(all_chunks)} chunks from {CHUNKS_FILE}")

In [ ]:
# only run manually when need to wipe and rebuild
# delete existing collection to force re-embedding
#try:
#    client = chromadb.PersistentClient(path=CHROMA_DIR)
#    client.delete_collection(COLLECTION_NAME)
#    print("Old collection deleted. Ready to re-embed.")
#except Exception as e:
#    print(f"Nothing to delete: {e}")

In [14]:
os.makedirs(CHROMA_DIR, exist_ok=True)

client     = chromadb.PersistentClient(path=CHROMA_DIR)


collection = client.get_or_create_collection(
    name     = COLLECTION_NAME,
    metadata = {"hnsw:space": "cosine"}

)
print(f"Collection '{COLLECTION_NAME}' ready.")
print(f"Currently contains {collection.count()} embeddings.")

Collection 'governance_rag_papers' ready.
Currently contains 1949 embeddings.


In [ ]:
print(f"Loading embedding model: {EMBEDDING_MODEL}")
model = SentenceTransformer(EMBEDDING_MODEL)
print("Model loaded.")

In [ ]:
# get all chunk_ids already stored in ChromaDB
existing_ids = set(collection.get(include=[])["ids"])
print(f"Already embedded: {len(existing_ids)} chunks")

# filter to only chunks that aren't embedded yet
new_chunks = [c for c in all_chunks if c["chunk_id"] not in existing_ids]
print(f"New chunks to embed: {len(new_chunks)}")

if len(new_chunks) == 0:
    print("Nothing to do — all chunks already embedded!")
else:

    total_batches = (len(new_chunks) + BATCH_SIZE - 1) // BATCH_SIZE

    for i in tqdm(range(0, len(new_chunks), BATCH_SIZE),
                  desc="Embedding chunks", total=total_batches):

        batch = new_chunks[i : i + BATCH_SIZE]


        texts = [c["text"] for c in batch]


        embeddings = model.encode(texts, show_progress_bar=False).tolist()

        # store in ChromaDB — three things per chunk:
        # 1. ids       — unique identifier for each chunk
        # 2. embeddings — the vectors we just computed
        # 3. documents  — the original text (for retrieval later)
        # 4. metadatas  — title, filename etc (for citations later)
        collection.add(
            ids        = [c["chunk_id"]  for c in batch],
            embeddings = embeddings,
            documents  = [c["text"]      for c in batch],
            metadatas  = [
                {
                    "title":     c["title"],
                    "filename":  c["filename"],
                    "chunk_idx": c["chunk_idx"],
                }
                for c in batch
            ]
        )

    print(f"\nDone. Total embeddings in DB: {collection.count()}")

In [ ]:
test_question = "What are the main challenges of Governance AI?"

# embed the question the same way we embedded the chunks
question_embedding = model.encode(test_question).tolist()

# query ChromaDB for top 3 most similar chunks
results = collection.query(
    query_embeddings = [question_embedding],
    n_results        = 3,
    include          = ["documents", "metadatas", "distances"]
)

print(f"Test query: '{test_question}'\n")
print("Top 3 results:\n")

for i, (doc, meta, dist) in enumerate(zip(
    results["documents"][0],
    results["metadatas"][0],
    results["distances"][0]
), 1):
    # distance is cosine distance — lower = more similar
    # we convert to similarity score: 1 - distance
    similarity = round(1 - dist, 3)
    print(f"Result {i} — similarity: {similarity}")
    print(f"From: {meta['title']}")
    print(f"Text: {doc[:200]}...")
    print()

In [ ]:
print("Loading embedding model...")
embedder = SentenceTransformer(EMBEDDING_MODEL)

# Load the cross-encoder for reranking

print("Loading reranking model...")
reranker = CrossEncoder(RERANK_MODEL)

# Connect persistent ChromaDB
client     = chromadb.PersistentClient(path=CHROMA_DIR)
collection = client.get_collection(COLLECTION_NAME)

print(f"Connected to ChromaDB — {collection.count()} chunks indexed.")
print("All models loaded.")

In [ ]:
# Takes a question, returns top RETRIEVE_N chunks from ChromaDB

def retrieve(question, n=RETRIEVE_N):
    """
    Embed the question and find the most similar chunks
    in ChromaDB using cosine similarity.
    Returns a list of dicts with text, title, filename, score.
    """
    # embed the question — same model as chunks
    question_embedding = embedder.encode(question).tolist()

    # query ChromaDB
    results = collection.query(
        query_embeddings = [question_embedding],
        n_results        = n,
        include          = ["documents", "metadatas", "distances"]
    )

    # package results into clean dicts
    chunks = []
    for doc, meta, dist in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    ):
        chunks.append({
            "text":       doc,
            "title":      meta["title"],
            "filename":   meta["filename"],
            "similarity": round(1 - dist, 3),
        })

    return chunks

In [ ]:
# Takes the retrieved chunks and reranks them using a
# cross-encoder that scores (question, chunk) pairs together

def rerank(question, chunks, top_k=TOP_K, max_per_paper=MAX_PER_PAPER):
    """
    Rerank chunks using a cross-encoder model.
    Also enforces diversity — max_per_paper chunks per paper.
    Returns top_k most relevant chunks.
    """
    if not chunks:
        return []

    # create (question, chunk_text) pairs for the cross-encoder
    pairs = [(question, c["text"]) for c in chunks]

    # cross-encoder scores each pair — higher = more relevant
    # unlike embedding similarity, this score is not bounded 0-1
    # it's a raw relevance score, higher is better
    scores = reranker.predict(pairs)

    # attach scores to chunks
    for chunk, score in zip(chunks, scores):
        chunk["rerank_score"] = round(float(score), 3)

    # sort by rerank score descending
    ranked = sorted(chunks, key=lambda x: x["rerank_score"], reverse=True)

    # enforce diversity — max max_per_paper chunks per paper
    # without this, one highly relevant paper could fill all top_k spots
    paper_counts = {}
    diverse      = []

    for chunk in ranked:
        paper = chunk["filename"]
        count = paper_counts.get(paper, 0)
        if count < max_per_paper:
            diverse.append(chunk)
            paper_counts[paper] = count + 1
        if len(diverse) == top_k:
            break

    return diverse

In [ ]:
# Takes the top reranked chunks and sends them to the LLM
# with the question to generate a grounded answer with citations

def generate(question, chunks, groq_api_key=GROQ_API_KEY):
    """
    Send retrieved chunks as context to the LLM.
    The LLM is instructed to answer ONLY from the provided context
    and cite the papers it uses.
    """
    # build the context string from chunks
    # each chunk is labeled with its source paper
    context_parts = []
    for i, chunk in enumerate(chunks, 1):
        context_parts.append(
            f"[Source {i}: {chunk['title']}]\n{chunk['text']}"
        )
    context = "\n\n---\n\n".join(context_parts)

    # the system prompt is critical in RAG
    # it tells the LLM to:
    # 1. answer ONLY from the provided context (prevents hallucination)
    # 2. cite sources by number (enables traceability)
    # 3. admit when it doesn't know (prevents making things up)
    system_prompt = """You are a research assistant specializing in Governance AI.
Answer the user's question using ONLY the provided context from research papers.
Always cite your sources using [Source N] notation.
If the context does not contain enough information to answer the question, say so clearly.
Do not use any knowledge outside of the provided context."""

    user_prompt = f"""Context from research papers:

{context}

Question: {question}

Answer with citations:"""

    # call Groq API
    client   = Groq(api_key=groq_api_key)
    response = client.chat.completions.create(
        model    = GROQ_MODEL,
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
        ],
        temperature = 0.1,  # low temperature = more factual, less creative
        max_tokens  = 1024,
    )

    return response.choices[0].message.content

In [ ]:
# Combines retrieve + rerank + generate into one clean function
# This is what Streamlit app will call
def ask(question, verbose=True):
    """
    Full RAG pipeline:
    1. Retrieve top 20 chunks from ChromaDB
    2. Rerank to top 5 with diversity enforcement
    3. Generate answer with citations
    """
    if verbose:
        print(f"Question: {question}\n")

    # step 1: retrieve
    retrieved = retrieve(question)
    if verbose:
        print(f"Retrieved {len(retrieved)} chunks.")

    # step 2: rerank
    reranked = rerank(question, retrieved)
    if verbose:
        print(f"Reranked to {len(reranked)} chunks.")
        print("\nSources used:")
        for i, c in enumerate(reranked, 1):
            print(f"  [{i}] {c['title']} (rerank score: {c['rerank_score']})")

    # step 3: generate
    if verbose:
        print("\nGenerating answer...\n")
    answer = generate(question, reranked)

    if verbose:
        print("=" * 60)
        print(answer)
        print("=" * 60)

    return {
        "question": question,
        "answer":   answer,
        "sources":  reranked,
    }

In [ ]:
# Test with a few real questions before building the UI

test_questions = [
    "What are the main challenges of Governance AI?",
    "How does bias in AI systems affect hiring decisions?",
    "What is the relationship between explainability and accountability in AI?",
]

for question in test_questions:
    result = ask(question)
    print("\n")

In [ ]:
stress_tests = [
    # hallucination tests
    "What did the EU AI Act specifically mandate in Article 13?",
    "What did Geoffrey Hinton say about Governance AI in 2024?",
    "What are the exact penalty amounts for violating AI fairness regulations in Germany?",

    # multi-hop reasoning
    "How does the lack of explainability in deep learning models create accountability gaps in high-stakes decisions?",
    "What is the relationship between data governance and algorithmic fairness?",
    "How do power imbalances between AI developers and affected communities undermine Governance AI principles?",

    # vague questions
    "Is AI good or bad?",
    "What should companies do about AI?",
    "How do we fix bias?",

    # out of scope
    "What is the best programming language for building AI systems?",
    "What is the capital of France?",

    # conflicting information
    "Is transparency always beneficial in AI systems?",
    "Can AI ever be truly fair?",
    "Is explainability sufficient for trustworthy AI?",
]

for question in stress_tests:
    print(f"\n{'='*60}")
    result = ask(question, verbose=False)  # verbose=False for cleaner output
    print(f"Q: {question}")
    print(f"\nSources: {[c['title'] for c in result['sources']]}")
    print(f"\nA: {result['answer']}")